# Create a new project and ingest DLC data

1. [Initiate Project](#Initiate-Project)
2. [Concatenating all DLC results](#Concatenating-all-DLC-results)
3. [Preprocessing](#Preprocessing)
4. [Create Velocity column](#Create-Velocity-column)
5. [Save Preprocessed Combined & Individual csvs](#Save-Preprocessed-Combined-&-Individual-csvs)


# Initiate Project

In [ ]:
from pathlib import Path
from datetime import datetime
import yaml
import os
import sys

custom_utils_path = Path(os.getcwd()).parent/'src'/'compass_labyrinth'/'behavior'/'pose_estimation'
sys.path.append(str(custom_utils_path))
import dlc_utils

project_path_full = Path(r'PATH\TO\PROJECT\compass_analysis')
user_metadata_file_path = r'PATH\TO\PROJECT\metadata.xlsx'
trial_type = 'YOUR_TRIAL_TYPE'
dlc_scorer = 'YOUR_DLC_SCORER'

cohort_metadata = dlc_utils.import_cohort_metadata(user_metadata_file_path, trial_type)
dlc_utils.validate_metadata(cohort_metadata)
cohort_metadata.to_csv(project_path_full / "cohort_metadata.csv", index=False)

session_names = [f"Session{int(s):04d}" for s in cohort_metadata["session"]]

config = {
    "project_name": "YOUR_PROJECT",
    "project_path_full": str(project_path_full),
    "creation_date_time": datetime.now().isoformat(),
    "trial_type": trial_type,
    "file_ext": ".csv",
    "video_type": ".mp4",
    "dlc_scorer": dlc_scorer,
    "session_names": session_names,
    "bodyparts": ["Nose", "Sternum", "Belly", "Tailbase", "FLPaw", "FRPaw", "BLPaw", "BRPaw"],
    "experimental_groups": ["GROUP_A", "GROUP_B"],
    "palette": "grey",
    "dlc_results_path": str(project_path_full / "dlc_results"),
}

with open(project_path_full / "config.yaml", "w") as f:
    yaml.dump(config, f)

print(f"Config built for {len(session_names)} sessions.")

In [ ]:
print(config.keys())
config

# Concatenating all DLC results

In [ ]:
from compass_labyrinth.behavior.preprocessing import compile_mouse_sessions


df_comb = compile_mouse_sessions(
    config=config,
    bp='Belly',
)
df_comb

# Preprocessing

In [ ]:
from compass_labyrinth.behavior.preprocessing import preprocess_sessions


df_all_csv = preprocess_sessions(df_comb=df_comb)
df_all_csv

# Create Velocity column

In [ ]:
from compass_labyrinth.behavior.preprocessing import ensure_velocity_column

df_all_csv = ensure_velocity_column(df_all_csv, fps=5)

# Save Preprocessed Combined & Individual csvs

In [ ]:
from compass_labyrinth.behavior.preprocessing import save_preprocessed_to_csv

save_preprocessed_to_csv(
    config=config,
    df=df_all_csv,
    save_combined=True,
    save_individual=True,
)